In [1]:
import numpy as np

In [4]:
np.where(np.array([0, 1, 1]))[0]

array([1, 2])

In [ ]:
from qldpc.codes.surgery import load_webster_seed_set
from qldpc.codes.common import CSSCode

def _build_generalised_bicycle_code(l: int, A_set: list[int], B_set: list[int]) -> CSSCode:
    """Build a generalised bicycle code from cyclic exponent sets A, B.

    Per Kovalev-Pryadko (arXiv:1212.6703) and Swaroop's reference
    implementation (https://github.com/eswaroop/adapters-LDPC-surgery,
    ext/bivariate_bicyclic.py): given subsets A, B of Z_l, let A(x) =
    sum(x^a for a in A_set) and B(x) = sum(x^b for b in B_set) as cyclic
    matrices in F_2[Z_l]. Then H_X = [A | B] and H_Z = [B^T | A^T] define
    the bicycle code on 2l data qubits.

    Args:
        l: cyclic group order.
        A_set, B_set: subsets of {0, 1, ..., l-1}.

    Returns:
        CSSCode on 2l data qubits with check matrices [A | B] and
        [B^T | A^T] over GF(2).
    """
    I_l = np.eye(l, dtype=np.int_)
    # cyclic shift matrix S such that S^k is left-shift by k (zero-indexed)
    S = np.roll(I_l, shift=-1, axis=0)
    A = np.zeros((l, l), dtype=np.int_)
    for a in A_set:
        A = (A + np.linalg.matrix_power(S, a)) % 2
    B = np.zeros((l, l), dtype=np.int_)
    for b in B_set:
        B = (B + np.linalg.matrix_power(S, b)) % 2

    H_X = np.hstack([A, B])
    H_Z = np.hstack([B.T, A.T])

    return CSSCode(H_X, H_Z, is_subsystem_code=False)


V0 = (1, 6, 8, 10, 42, 57)
# Pick Webster Appendix A code 0: l=31, n=62, k=10, d=6
data = load_webster_seed_set(0)
code = _build_generalised_bicycle_code(data["l"], data["A"], data["B"])
print(f"Code: [[{code.num_qudits}, {code.dimension}]]  (l={data['l']})")
print(f"Code name from JSON: {data.get('name', 'N/A')}")

def x_bar_1_operator(d: dict) -> np.ndarray:
    """Extract X̄_1 from a Webster seed_set dict as a 2l binary vector."""
    l = d["l"]
    for seed in d["seeds"]:
        if seed["name"] == "X_bar_1" and seed["pauli_type"] == "X":
            L = np.zeros(l, dtype=np.uint8)
            R = np.zeros(l, dtype=np.uint8)
            for i in seed["L_support"]:
                L[i] = 1
            for i in seed["R_support"]:
                R[i] = 1
            return np.concatenate([L, R])
    raise ValueError("X_bar_1 not found")

HZ = code.matrix_z
print(HZ.shape)
print(HZ[1, list(V0)].any())
C0 = tuple(int(j) for j in range(HZ.shape[0]) if HZ[j, list(V0)].any())

Code: [[62, 10]]  (l=31)
Code name from JSON: 62_10_6
(31, 62)
False
